In [ ]:
from functools import partial
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt

sys.path.append("../")
from src.data_generation.grid import Grid
from src.real_data.dataloader import temporal_split, build_task, make_real_eval_set
from src.model.finetune import finetune
from src.model.quote_loss import quote_arb_loss
from src.model.preprocessed_dataset import preprocess_surfaces
from src.evaluation import surface_eval as SE
from src.evaluation.surface_eval import check_arbitrage_flat
from src.model.SSVI import fit_ssvi, predict_ssvi

cfg = yaml.safe_load(open("../config.yaml"))
g = Grid(cfg)
n_grid = len(g.z)

In [ ]:
START = "2023-01-01"
END = "2024-01-01"
train_s, val_s, test_s = temporal_split(START, END, val_months=1, test_months=3)

day = lambda s: s["date"].iloc[0].date()
for name, p in [("train", train_s), ("val", val_s), ("test", test_s)]:
    print(f"{name:5s} {len(p):3d} surfaces   {day(p[0])} -> {day(p[-1])}")

In [ ]:
RUN_NAME = "real_v1"
N_CONTEXT = (5, 60)
N_HELDOUT = 1500 
EVAL_SIZES = [5, 10, 20, 40]
GROUP_SIZE = 8

N_EPOCHS = 50
N_PER_EPOCH = 256
BATCH_SIZE = 16
VAL_EVERY = 5
LR = 1e-5

LAMBDA_CAL = 1.0
LAMBDA_BF = 1.0

In [ ]:
train_provider = partial(build_task, train_s, n_context=N_CONTEXT, grid=g, n_heldout=N_HELDOUT, size_group=GROUP_SIZE)
val_data = make_real_eval_set(val_s, EVAL_SIZES, grid=g,n_heldout=N_HELDOUT)
loss_fn = partial(quote_arb_loss, grid_shape=g.shape, lambda_cal=LAMBDA_CAL, lambda_bf=LAMBDA_BF)

In [ ]:
model = finetune(
    train_provider,
    RUN_NAME,
    n_epochs=N_EPOCHS,
    n_surfaces_per_epoch=N_PER_EPOCH,
    batch_size=BATCH_SIZE,
    val_data=val_data,
    val_every=VAL_EVERY,
    loss_fn=loss_fn,
    lr=LR,
)

In [ ]:
QS = [0.025, 0.05, 0.1, 0.25, 0.75, 0.9, 0.95, 0.975]


def run_predictions(state, train_list, test_list):
    est = SE._get_eval_estimator()
    est.model_.load_state_dict(state if state is not None else SE._pretrained_state)
    surfaces = preprocess_surfaces(est, train_list, test_list, np.random.default_rng(0), group_size=1)
    recs = []
    with torch.no_grad():
        for i, s in enumerate(surfaces):
            print(f"task {i+1}/{len(surfaces)}", end="\r", flush=True)
            est.raw_space_bardist_ = s.raw_space_bardist
            est.znorm_space_bardist_ = s.znorm_space_bardist
            est.fit_from_preprocessed(s.X_context, s.y_context, s.cat_indices, s.configs,
                                      performance_options=SE._PERF, no_refit=True)
            _, ple, _ = est.forward(s.X_query, use_inference_mode=False)
            L = torch.stack(ple, dim=2)
            Q, B, E, Ld = L.shape
            LBQ = L.permute(1, 2, 0, 3).reshape(B * E, Q, Ld)
            for gi in range(B):
                bd, lg = s.raw_bardists[gi], LBQ[gi * E:(gi + 1) * E]
                recs.append(dict(
                    mean=bd.mean(lg)[0].cpu().numpy(),
                    std=bd.variance(lg)[0].sqrt().cpu().numpy(),
                    q={p: bd.icdf(lg, p)[0].cpu().numpy() for p in QS},
                    y_te=s.y_query_raw[gi].numpy(),
                    zt=s.X_query_raw[gi][:, :2].numpy(),
                ))
    return recs


state = torch.load(Path("../checkpoints") / RUN_NAME / "final.pt", map_location="cpu")
et_tr, et_te = make_real_eval_set(test_s, EVAL_SIZES, grid=g)
recs = run_predictions(state, et_tr, et_te)
for r, (_, y_ctx) in zip(recs, et_tr):
    r["n"] = len(y_ctx) // 2
print(len(recs), "test tasks")

In [ ]:
for i, (r, (X_tr, y_tr), (X_te, y_te)) in enumerate(zip(recs, et_tr, et_te)):
    print(f"task {i+1}/{len(recs)}", end="\r", flush=True)
    nc = len(y_tr) // 2
    z, tau = X_tr[:nc, 0], X_tr[:nc, 1]
    mid = (y_tr[:nc] + y_tr[nc:]) / 2
    params, _ = fit_ssvi(np.column_stack([z * np.sqrt(tau), tau]), mid, cfg)
    m = np.isfinite(y_te).all(1)
    zh, th = X_te[m, 0], X_te[m, 1]
    r["ssvi"] = predict_ssvi(params, th, zh * np.sqrt(th))

In [ ]:
rows = []
for r in recs:
    m = np.isfinite(r["y_te"]).all(1)
    mid = r["y_te"][m].mean(1)
    rows.append((r["n"],
                 np.abs(r["ssvi"] - mid).mean() * 100,
                 np.abs(r["mean"][m] - mid).mean() * 100))
df = pd.DataFrame(rows, columns=["N", "SSVI MAE%", "model MAE%"]).groupby("N").mean().round(3)
print(df.to_string())

In [ ]:
rows = []
for r in recs:
    m = np.isfinite(r["y_te"]).all(1)
    bid, ask = r["y_te"][m, 0], r["y_te"][m, 1]
    rows.append((r["n"],
                 ((r["ssvi"] >= bid) & (r["ssvi"] <= ask)).mean(),
                 ((r["mean"][m] >= bid) & (r["mean"][m] <= ask)).mean()))
df = pd.DataFrame(rows, columns=["N", "SSVI inside", "model inside"]).groupby("N").mean().round(3)
print(df.to_string())

In [ ]:
# arbitrage violation rate
rows = []
for r in recs:
    cal, bf = check_arbitrage_flat(cfg, r["mean"][:n_grid])
    rows.append((r["n"], cal * 100, bf * 100))
print(pd.DataFrame(rows, columns=["n", "cal", "bf"]).groupby("n")[["cal", "bf"]].mean().round(1).to_string())

In [ ]:
LEVELS = {0.5: (0.25, 0.75), 0.8: (0.1, 0.9), 0.9: (0.05, 0.95), 0.95: (0.025, 0.975)}
rows = []
for r in recs:
    m = np.isfinite(r["y_te"]).all(1)
    mid = r["y_te"][m].mean(1)
    row = {"n": r["n"], "std%": r["std"][m].mean() * 100}
    for lv, (lo, hi) in LEVELS.items():
        row[f"cov{int(lv * 100)}"] = ((mid >= r["q"][lo][m]) & (mid <= r["q"][hi][m])).mean()
    rows.append(row)
print(pd.DataFrame(rows).groupby("n").mean().round(3).to_string())